In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import Lasso, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import RFE

# 실행되지 않는다면 관련 클래스 설치 필요

## EDA & 피처 엔지니어링 과제

**데이터셋**: `seaborn.load_dataset('titanic')` — 별도 파일 업로드 없이 바로 불러와서 사용합니다.

In [ ]:
# 데이터 로드
df = sns.load_dataset('titanic')

In [ ]:
# 데이터 확인
df.head()

In [ ]:
# 데이터 기본 정보 출력
print(df.info())
print(df.describe())

## **기초 통계 분석**

In [ ]:
# 기초 통계 분석
print("평균:\n", df[['age','fare']].mean())
print("중앙값:\n", df[['age','fare']].median())
print("최빈값:\n", df[['age','fare']].mode().iloc[0])

### **데이터 시각화**

Q1. 시각화 방법을 보고 _____에 관련 코드를 적어주세요

In [ ]:
# 산점도
# 나이와 요금 간의 관계 확인
plt.______(df['age'], df['fare'])
plt.xlabel('Age'); plt.ylabel('Fare')
plt.show()

In [ ]:
# 히스토그램
f_a = df.loc[df.sex=='female','age']
m_a = df.loc[df.sex=='male','age']
plt.______(f_a.dropna(), bins=20, alpha=0.5, label='female')
plt.______(m_a.dropna(), bins=20, alpha=0.5, label='male')
plt.legend()
plt.show()

In [ ]:
# 박스플롯 (이상치 확인)
sns.______(data=df, x='pclass', y='fare', hue='survived')
plt.show()

In [ ]:
# 히트맵
numcols = ['survived','pclass','age','sibsp','parch','fare']
sns.______(df[numcols].corr(), annot=True)
plt.show()

In [ ]:
# pairplot
sns.pairplot(df[numcols].dropna())

In [ ]:
# 바이올린 플랏
plt.figure(figsize=(12, 6))
sns.______(data=df[['age','fare','sibsp','parch']])
plt.title("Violin Plot of Key Titanic Features")
plt.show()

In [ ]:
# 밀도 플롯 (Kernel Density Estimation, KDE)
plt.figure(figsize=(12, 6))
sns.______(df['fare'], fill=True, color='red', label='Fare')
sns.______(df['age'], fill=True, color='blue', label='Age')
plt.title("Density Plot of Fare & Age")
plt.legend()
plt.show()

### **이상치**

Q2. 주제를 보고 빈칸에 적절한 코드를 작성해 완성시켜 주세요

In [ ]:
# 이상치 탐색 (IQR 사용) - fare 기준
Q1 = df['fare'].______(0.25)
Q3 = df['fare'].______(0.75)
IQR = Q3 - Q1
outliers = ((df['fare'] < (Q1 - 1.5 * IQR)) | (df['fare'] > (Q3 + 1.5 * IQR)))
print("이상치 개수:", outliers.sum())

### **데이터 전처리**

Q3. 주제를 보고 빈칸에 적절한 코드를 작성해 완성시켜 주세요.

In [ ]:
# 결측치 처리
## SimpleImputer는 결측치를 채우는 방법을 제공하는 사이킷런의 클래스로,
## strategy='median'을 사용해 각 열의 중앙값으로 결측치를 채웁니다.
imputer = SimpleImputer(strategy='______')
df['age'] = imputer.fit_transform(df[['age']])

df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode()[0])

## 결측치가 너무 많은 deck 컬럼은 제거합니다.
df = df.drop(columns=['deck'])

# 이상치 제거 (앞에서 구한 Q1, Q3, IQR 활용)
df = df[~((df['fare'] < (Q1 - 1.5 * IQR)) | (df['fare'] > (Q3 + 1.5 * IQR)))]

# 범주형 데이터 처리
## get_dummies를 사용하여 범주형 변수를 원-핫 인코딩합니다.
## drop_first=True를 사용하여 첫번째 변수 제거 -> 더미변수 함정 방지
cat_cols = ['sex', 'embarked']
df = pd.______(df, columns=cat_cols, drop_first=True)

In [ ]:
# 데이터 전처리 확인
df.info()

In [ ]:
df.head()

### **변수 선택 (랜덤포레스트 기반)**

In [ ]:
# y에 가장 큰 영향을 주는 x를 분석하는 것이 목표

# 랜덤포레스트 작동 원리
#1. 데이터 샘플링 (여러개의 결정트리 생성)
#2. 트리학습 (샘플링된 데이터와 일부 랜덤 선택된 feature만 사용하여 학습)
#3. 결과 평균화 (여러 트리의 결과를 다수결/평균으로 최종 예측)

In [ ]:
# 데이터 선택
x = df[['pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_male']]
y = df['survived']

# 모델 학습
model = RandomForestClassifier(random_state=42)
model.fit(x, y)

# 변수 중요도 계산 코드
## model 학습을 하면 feature_importances_ 속성 안에 변수중요도값이 저장되어 있습니다.
feature_importances = pd.Series(model.feature_importances_, index=x.columns)

# 출력
print("변수 중요도:\n", feature_importances.sort_values(ascending=False))

Q4. 결과값을 붙여놓고 어떤 변수가 'survived'와 가장 높은 상관관계를(중요도를) 갖는지 적어주세요.

In [ ]:
# 결과값:
# 가장 중요한 변수:

### **차원 축소 & 스케일링**

Q5. 코드 내에 제시되어 있는 문제에 따라 코드를 작성해주세요.

In [ ]:
# 데이터 스케일링 (표준화)
## 평균=0, 분산=1로 변환시키는 코드를 빈칸에 작성해주세요.
scaler = ______()
X_scaled = scaler.fit_transform(x)

# PCA 객체 생성 및 차원 축소
## n_components는 몇 개의 주성분으로 차원을 축소할 건지 결정합니다.
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# 출력
print("PCA 설명된 분산 비율:", pca.explained_variance_ratio_)

Q6. 아래 빈칸을 채워 출력값의 의미를 적어주세요.

A. 첫 번째 주성분(PC1)이 전체 데이터 ___의 ___%를 설명하고, 두 번째 주성분은 ___%를 설명합니다.

### **파생 변수**

In [ ]:
# 파생 변수 생성 예시
## family_size = 형제/배우자 수(sibsp) + 부모/자녀 수(parch) + 본인(1)
df['family_size'] = df['sibsp'] + df['parch'] + 1

## is_alone = 혼자 탑승했는지 여부
df['is_alone'] = (df['family_size'] == 1).astype(int)

## fare_per_person = 1인당 지불한 요금
df['fare_per_person'] = df['fare'] / df['family_size']

In [ ]:
df[['family_size', 'is_alone', 'fare_per_person', 'fare']].head()

Q7. 원하는 파생변수를 직접 생성해주세요.
  예시: 연령대(age_group), 등급×성별 조합, 요금 구간(fare_bin) 등

In [ ]:
# Q8. 여기에 나만의 파생변수 생성 코드를 작성하세요.
